In [1]:
!uv pip install faiss-cpu

⠼                                                                               Resolved 3 packages in 640ms
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)--------------     0 B/29.85 MiB           
⠙ Preparing packages... (0/1)-------------- 14.91 KiB/29.85 MiB         
⠙ Preparing packages... (0/1)-------------- 30.91 KiB/29.85 MiB         
⠙ Preparing packages... (0/1)-------------- 46.91 KiB/29.85 MiB         
⠙ Preparing packages... (0/1)-------------- 62.91 KiB/29.85 MiB         
⠙ Preparing packages... (0/1)-------------- 78.91 KiB/29.85 MiB         
⠙ Preparing packages... (0/1)-------------- 94.91 KiB/29.85 MiB         
⠙ Preparing packages... (0/1)-------------- 110.91 KiB/29.85 MiB        
⠙ Preparing packages... (0/1)-------------- 126.91 KiB/29.85 MiB        
⠙ Preparing packages... (0/1)-------------- 142.91 KiB/29.85 MiB        
⠙ Preparing packages... (0/1)-------------- 158.91 KiB/29.85 MiB        
⠙ Prepa

In [1]:
# GPU-Optimized Multimodal RAG with ColPali + MedGemma - Complete Implementation
# Enhanced for Leishmania Research with Peer Suggestions Applied

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # helps determinism on some GPUs
# Set up paths and environment (unified with Code A)
os.environ["HF_HOME"] = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf"
os.environ["HF_HUB_CACHE"] = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf/transformers"
import json
import time
import logging
import gc
import math
import hashlib
import threading
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import faiss


# Unified path configuration
PROJECT_DIR = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
A_RAG_DIR = PROJECT_DIR / "kaggle" / "working" / "rag_knowledge_base"
A_IMG_DIR = A_RAG_DIR / "images"
A_META_DIR = A_RAG_DIR / "metadata"
INDEX_DIR = A_RAG_DIR / "index"
COLQWEN2_DIR = INDEX_DIR / "colqwen2"
PAGE_RENDERS_DIR = A_RAG_DIR / "page_renders"

# Create directories
for d in [A_RAG_DIR, A_IMG_DIR, A_META_DIR, INDEX_DIR, COLQWEN2_DIR, PAGE_RENDERS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# GPU configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --- Determinism & config ---
import random
SEED = 1337

def set_determinism(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Call this once at startup
set_determinism()
DEBUG_GPU_SPIN = False  # Disable GPU spinners by default

# Leishmania keywords for intelligent filtering
LEISHMANIA_KEYWORDS = [
    'leishmaniasis', 'leishmania', 'kala-azar', 'visceral leishmaniasis',
    'cutaneous leishmaniasis', 'mucocutaneous leishmaniasis',
    'sandfly', 'phlebotomus', 'lutzomyia', 'amastigotes', 'promastigotes',
    'montenegro test', 'pentavalent antimony', 'amphotericin b',
    'miltefosine', 'chiclero', 'espundia', 'oriental sore'
]
# --- Grounded prompting template ---
GROUND_RULES = """\
You are a careful medical assistant. Use ONLY the provided evidence (images and their doc_id:page) to answer.
Rules:
1) Stay strictly on topic (Leishmania and content present in evidence).
2) If the evidence is insufficient or unclear, output EXACTLY: INSUFFICIENT_EVIDENCE
3) Cite sources inline like [doc_id:page]. Do not invent citations.
4) Be concise, factual, and avoid speculation."""

def make_grounded_prompt(user_question: str, evidence_meta: list) -> str:
    # evidence_meta: list of dicts with keys 'doc_id' and 'page'
    cites = []
    for m in evidence_meta[:8]:  # show up to 8 refs in the prompt
        d = m.get("doc_id", "unknown")
        p = m.get("page", "?")
        cites.append(f"[{d}:{p}]")
    cites_str = ", ".join(cites) if cites else "(no citations available)"

    return (
        f"{GROUND_RULES}\n\n"
        f"Evidence citations available: {cites_str}\n\n"
        f"Question: {user_question}\n\n"
        f"Answer with citations, or write INSUFFICIENT_EVIDENCE."
    )

# Read model from manifest to match index
try:
    with open(COLQWEN2_DIR / "manifest.json") as f:
        manifest = json.load(f)
    DEFAULT_COLQWEN2_MODEL = manifest.get("model", "vidore/colqwen2-v1.0-hf")
except Exception:
    DEFAULT_COLQWEN2_MODEL = "vidore/colqwen2-v1.0-hf"

logger.info(f"Using device: {device}")
logger.info(f"Project directory: {PROJECT_DIR}")
logger.info(f"RAG directory: {A_RAG_DIR}")

# ============================================================================
# SECTION 1: ColQwen2 Page Renderer and Index Builder (A.5 step)
# ============================================================================

def render_pdf_pages(pdf_path: Path, output_dir: Path, dpi: int = 150) -> List[Dict[str, Any]]:
    """Render PDF pages to images with metadata tracking."""
    try:
        from pdf2image import convert_from_path
        import fitz  # PyMuPDF
        
        doc_id = pdf_path.stem
        doc_output_dir = output_dir / doc_id
        doc_output_dir.mkdir(exist_ok=True)
        
        # Convert PDF to images
        pages = convert_from_path(
            str(pdf_path), 
            dpi=dpi,
            fmt='PNG',
            thread_count=4
        )
        
        page_metadata = []
        for i, page in enumerate(pages):
            page_num = i + 1
            img_filename = f"page_{page_num:04d}.png"
            img_path = doc_output_dir / img_filename
            
            # Optimize image for processing
            if page.mode != 'RGB':
                page = page.convert('RGB')
            if max(page.size) > 2048:
                page.thumbnail((2048, 2048), Image.Resampling.LANCZOS)
                
            page.save(img_path, "PNG", optimize=True)
            
            page_metadata.append({
                "source_file": str(pdf_path),
                "doc_id": doc_id,
                "page": page_num,
                "image_path": str(img_path),
                "width": page.width,
                "height": page.height,
                "row_id": len(page_metadata)  # Stable row ID for indexing
            })
            
        logger.info(f"Rendered {len(pages)} pages from {pdf_path.name}")
        return page_metadata
        
    except Exception as e:
        logger.error(f"Error rendering {pdf_path}: {e}")
        return []

def build_colqwen2_index():
    """Build ColQwen2 index with packed embeddings and FAISS fallback."""
    from transformers import AutoProcessor, AutoModel
    from transformers.utils import is_flash_attn_2_available
    
    MODEL_NAME = "vidore/colqwen2-v1.0-hf"
    DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    BATCH_SIZE = 4
    
    logger.info(f"Loading ColQwen2 model: {MODEL_NAME}")
    
    # Load model with proper configuration
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=DTYPE,
        device_map="auto" if torch.cuda.is_available() else None,
        attn_implementation="eager",  # Stable implementation
        trust_remote_code=True
    )
    
    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    
    if not torch.cuda.is_available():
        model = model.to(device)
    
    model.eval()
    
    # Load image catalog
    def load_image_catalog() -> List[Dict[str, Any]]:
        items = []
        
        # Try to load from existing metadata
        if A_META_DIR.exists():
            for p in sorted(A_META_DIR.glob("*_images.json")):
                try:
                    data = json.loads(p.read_text(encoding="utf-8"))
                    for row in data:
                        if row.get("image_path") and Path(row["image_path"]).exists():
                            items.append(row)
                except Exception as e:
                    logger.warning(f"Error loading {p.name}: {e}")
        
        # Fallback: scan page renders directory
        if not items and PAGE_RENDERS_DIR.exists():
            for img_path in sorted(PAGE_RENDERS_DIR.rglob("*.png")):
                items.append({
                    "source_file": None,
                    "doc_id": img_path.parent.name,
                    "page": int(img_path.stem.split('_')[-1]) if '_' in img_path.stem else 1,
                    "image_path": str(img_path),
                    "row_id": len(items)
                })
        
        return items
    
    catalog = load_image_catalog()
    
    if not catalog:
        logger.warning("No images found. Creating demo content...")
        # Create demo content for testing
        demo_img = Image.new('RGB', (400, 300), color='white')
        demo_path = PAGE_RENDERS_DIR / "demo" / "page_0001.png"
        demo_path.parent.mkdir(exist_ok=True)
        demo_img.save(demo_path)
        
        catalog = [{
            "source_file": "demo.pdf",
            "doc_id": "demo",
            "page": 1,
            "image_path": str(demo_path),
            "row_id": 0
        }]
    
    logger.info(f"Processing {len(catalog)} images")
    
    # Encode images in batches
    packed = []
    offsets = []
    avg_vecs = []
    start = 0
    dim = None
    
    all_paths = [item["image_path"] for item in catalog]
    
    for i in range(0, len(all_paths), BATCH_SIZE):
        batch_paths = all_paths[i:i+BATCH_SIZE]
        
        try:
            # Load images
            images = []
            for path in batch_paths:
                with Image.open(path) as img:
                    images.append(img.convert("RGB"))
            
            # Process with ColQwen2
            inputs = processor(images=images, return_tensors="pt")
            if torch.cuda.is_available():
                inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs)
                embeddings = outputs.last_hidden_state  # [batch, seq_len, dim]
                
                for j, emb in enumerate(embeddings):
                    # Remove padding if present
                    if 'attention_mask' in inputs:
                        mask = inputs['attention_mask'][j]
                        emb = emb[mask.bool()]
                    
                    # Normalize for cosine similarity
                    emb = F.normalize(emb.float(), dim=-1)
                    
                    if dim is None:
                        dim = emb.shape[-1]
                    
                    seq_len = emb.shape[0]
                    
                    # Store packed embeddings
                    packed.append(emb.to(torch.float16).cpu().numpy())
                    
                    # Compute average embedding
                    avg_emb = F.normalize(emb.mean(dim=0, keepdim=True), dim=-1)[0]
                    avg_vecs.append(avg_emb.to(torch.float32).cpu().numpy())
                    
                    # Store offset
                    offsets.append((start, seq_len))
                    start += seq_len
                    
        except Exception as e:
            logger.error(f"Error processing batch {i//BATCH_SIZE + 1}: {e}")
            continue
            
        if (i // BATCH_SIZE + 1) % 10 == 0:
            logger.info(f"Processed {i + len(batch_paths)}/{len(all_paths)} images")
            
        # Memory cleanup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    
    # Save packed embeddings and metadata
    if packed:
        E = np.concatenate(packed, axis=0)
        AVG = np.stack(avg_vecs, axis=0).astype(np.float32)
        OFF = np.asarray(offsets, dtype=np.int64)
        
        # Save files
        E.tofile(COLQWEN2_DIR / "packed_embeddings.f16.npy")
        np.save(COLQWEN2_DIR / "avg_vectors.npy", AVG)
        np.save(COLQWEN2_DIR / "offsets.npy", OFF)
        
        # Save metadata with stable row_id
        with open(COLQWEN2_DIR / "meta.jsonl", "w", encoding="utf-8") as f:
            for row_id, item in enumerate(catalog):
                item_copy = dict(item)
                item_copy["row_id"] = row_id
                f.write(json.dumps(item_copy, ensure_ascii=False) + "\n")
        
        # Build FAISS index
        # Ensure L2-normalized
        row_norms = np.linalg.norm(AVG, axis=1, keepdims=True) + 1e-12
        AVG = AVG / row_norms
        
        index = faiss.IndexFlatIP(AVG.shape[1])
        index.add(AVG)
        faiss.write_index(index, str(INDEX_DIR / "image_avg.faiss"))
        
        # Save manifest
        manifest = {
            "created_at": datetime.now().isoformat(),
            "model": MODEL_NAME,
            "dtype": "float16 (packed), float32 (avg)",
            "dim": int(dim),
            "n_images": len(catalog),
            "n_tokens_total": int(E.shape[0]),
            "files": {
                "packed": str(COLQWEN2_DIR / "packed_embeddings.f16.npy"),
                "offsets": str(COLQWEN2_DIR / "offsets.npy"),
                "avg": str(COLQWEN2_DIR / "avg_vectors.npy"),
                "meta": str(COLQWEN2_DIR / "meta.jsonl"),
                "faiss": str(INDEX_DIR / "image_avg.faiss")
            }
        }
        
        with open(COLQWEN2_DIR / "manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        
        logger.info(f"✅ ColQwen2 index built successfully!")
        logger.info(f"   - {len(catalog)} images indexed")
        logger.info(f"   - {E.shape[0]:,} token embeddings")
        logger.info(f"   - Dimension: {dim}")
        
    else:
        logger.error("No embeddings generated!")

# ============================================================================
# SECTION 2: Query-Only ColQwen2 Encoder and Retrieval System
# ============================================================================

class ColQwen2QueryEncoder:
    """Lightweight ColQwen2 encoder for queries only."""
    
    def __init__(self, model_name: str = None):
        if model_name is None:
            model_name = DEFAULT_COLQWEN2_MODEL
        self.model_name = model_name
        self.device = device
        self.model = None
        self.processor = None
        
        # Lazy loading flags
        self._faiss_idx = None
        self._meta = None
        self._packed_data = None
        
    def _load_model(self):
        """Lazy load the model."""
        if self.model is None:
            from transformers import ColQwen2ForRetrieval, ColQwen2Processor
            from transformers.utils import is_flash_attn_2_available
            
            logger.info(f"Loading ColQwen2 query encoder: {self.model_name}")
            
            # dtype selection
            if torch.cuda.is_available():
                if torch.cuda.is_bf16_supported():
                    dtype = torch.bfloat16
                else:
                    dtype = torch.float16
            else:
                dtype = torch.float32
            
            attn_impl = "flash_attention_2" if is_flash_attn_2_available() else "eager"
            
            # Use HF-native classes
            self.model = ColQwen2ForRetrieval.from_pretrained(
                self.model_name,
                torch_dtype=dtype,
                device_map="auto" if torch.cuda.is_available() else None,
                attn_implementation=attn_impl,
                trust_remote_code=True
            )
            
            self.processor = ColQwen2Processor.from_pretrained(
                self.model_name,
                trust_remote_code=True
            )
            
            if not torch.cuda.is_available():
                self.model = self.model.to(self.device)
            
            self.model.eval()
    
    def _load_faiss(self):
        """Lazy load FAISS index."""
        if self._faiss_idx is None:
            faiss_path = INDEX_DIR / "image_avg.faiss"
            if faiss_path.exists():
                self._faiss_idx = faiss.read_index(str(faiss_path))
                logger.info(f"Loaded FAISS index: {faiss_path}")
            else:
                raise FileNotFoundError(f"FAISS index not found: {faiss_path}")
        return self._faiss_idx
    
    def _load_meta(self):
        """Lazy load metadata."""
        if self._meta is None:
            meta_path = COLQWEN2_DIR / "meta.jsonl"
            if meta_path.exists():
                self._meta = []
                with open(meta_path, "r", encoding="utf-8") as f:
                    for line in f:
                        self._meta.append(json.loads(line.strip()))
                logger.info(f"Loaded metadata for {len(self._meta)} images")
            else:
                raise FileNotFoundError(f"Metadata not found: {meta_path}")
        return self._meta
    
    def _load_packed(self):
        """Lazy load packed embeddings."""
        if self._packed_data is None:
            packed_path = COLQWEN2_DIR / "packed_embeddings.f16.npy"
            offsets_path = COLQWEN2_DIR / "offsets.npy"
            
            if packed_path.exists() and offsets_path.exists():
                # Load manifest to get dimensions
                manifest_path = COLQWEN2_DIR / "manifest.json"
                if manifest_path.exists():
                    with open(manifest_path) as f:
                        manifest = json.load(f)
                    dim = manifest["dim"]
                else:
                    dim = 1024  # Default fallback
                
                arr = np.fromfile(packed_path, dtype=np.float16).astype(np.float32)
                E = arr.reshape(-1, dim)
                OFF = np.load(offsets_path)
                
                self._packed_data = (E, OFF, dim)
                logger.info(f"Loaded packed embeddings: {E.shape}")
            else:
                raise FileNotFoundError("Packed embeddings not found")
        
        return self._packed_data
    
    def encode_query_tokens(self, text: str) -> torch.Tensor:
        """Encode query text to token embeddings."""
        self._load_model()
        
        with torch.no_grad():
            inputs = self.processor(text=[text], return_tensors="pt")
            if torch.cuda.is_available():
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            out = self.model(**inputs)
            # Handle different output field names
            T = None
            for name in ("embeddings", "last_hidden_state", "text_embeddings"):
                if hasattr(out, name):
                    val = getattr(out, name)
                    # collapse [1, T, d] -> [T, d] if needed
                    T = val[0] if isinstance(val, torch.Tensor) and val.dim() == 3 else val
                    break
            if T is None:
                raise RuntimeError("ColQwen2 output did not contain token embeddings")
            
            Q = F.normalize(T.float(), dim=-1)
        
        return Q
    
    def encode_query_avg(self, text: str) -> np.ndarray:
        """Encode query text to averaged embedding."""
        Q = self.encode_query_tokens(text)
        q_avg = F.normalize(Q.mean(dim=0, keepdim=True), dim=-1)
        return q_avg.cpu().numpy().astype(np.float32)
    
    def search_avg(self, text: str, k: int = 5) -> List[Dict[str, Any]]:
        """Fast search using averaged embeddings."""
        idx = self._load_faiss()
        meta = self._load_meta()
        
        q = self.encode_query_avg(text)
        D, I = idx.search(q, min(k, len(meta)))
        
        hits = []
        for r in range(I.shape[1]):
            if I[0, r] < len(meta):
                hit = {
                    "rank": r + 1,
                    "score": float(D[0, r]),
                    "preselect_ip": float(D[0, r]),
                    **meta[I[0, r]]
                }
                hits.append(hit)
        
        return hits
    
    def search_late_interaction(self, text: str, k: int = 5, preselect: int = 64) -> List[Dict[str, Any]]:
        """Precise search using late interaction (MaxSim)."""
        # First, preselect candidates using avg search
        candidates = self.search_avg(text, preselect)
        
        if not candidates:
            return []
        
        # Load packed data
        E, OFF, dim = self._load_packed()
        meta = self._load_meta()
        
        # Encode query
        Q = self.encode_query_tokens(text)  # [Tq, d]
        
        # keep IP next to the late score
        scores = []
        for cand in candidates:
            row_id = cand.get("row_id")
            if row_id is None or row_id >= len(OFF):
                continue
            start, length = OFF[row_id]
            if start < 0 or length <= 0 or start + length > E.shape[0]:
                continue
            P = torch.from_numpy(E[start:start+length]).to(Q.device)
            maxsim = torch.max(Q @ P.T, dim=1).values.sum().item()
            scores.append({
                **cand,
                "preselect_ip": float(cand["score"]),
                "score_maxsim": float(maxsim),
            })

        scores.sort(key=lambda x: -x["score_maxsim"])
        return [
            {**s, "rank": i+1, "score": s["score_maxsim"]}  # keep 'score' = MaxSim for display
            for i, s in enumerate(scores[:k])
        ]

# Initialize global query encoder
query_encoder = ColQwen2QueryEncoder()

# ============================================================================
# SECTION 3: MedGemma Answer Generation
# ============================================================================

class GPUOptimizedMedGemma:
    """GPU-optimized MedGemma for medical answer generation."""
    
    def __init__(self, model_id: str = "google/medgemma-4b-it"):
        self.model_id = model_id
        self.device = device
        self.model = None
        self.processor = None
        self.generation_counter = 0
        
    def _load_model(self):
        """Lazy load MedGemma model."""
        if self.model is None:
            from transformers import AutoProcessor, AutoModelForImageTextToText
            
            logger.info(f"Loading MedGemma: {self.model_id}")
            
            try:
                self.processor = AutoProcessor.from_pretrained(
                    self.model_id,
                    trust_remote_code=True
                )
                
                if torch.cuda.is_available():
                    self.model = AutoModelForImageTextToText.from_pretrained(
                        self.model_id,
                        torch_dtype=torch.bfloat16,
                        device_map="auto",
                        trust_remote_code=True,
                        attn_implementation="eager"
                    )
                else:
                    self.model = AutoModelForImageTextToText.from_pretrained(
                        self.model_id,
                        torch_dtype=torch.float32,
                        trust_remote_code=True
                    )
                    self.model = self.model.to(self.device)
                
                self.model.eval()
                
                # Configure tokenizer
                if self.processor.tokenizer.pad_token is None:
                    self.processor.tokenizer.pad_token = self.processor.tokenizer.eos_token
                
                logger.info(f"✅ MedGemma loaded on {self.device}")
                
            except Exception as e:
                logger.error(f"Failed to load MedGemma: {e}")
                raise
    
    def generate_answer(self, query: str, images: List[Image.Image], max_images: int = 3) -> str:
        """Generate medical answer from query and images."""
        self._load_model()
        
        try:
            # Limit images for memory efficiency
            processed_images = images[:max_images] if images else []
            
            logger.info(f"Generating answer with {len(processed_images)} images")
            
            # Prepare messages
            if processed_images:
                messages = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": query}
                        ] + [{"type": "image"} for _ in processed_images]
                    }
                ]
            else:
                messages = [{"role": "user", "content": query}]
            
            # Apply chat template
            if hasattr(self.processor, 'apply_chat_template'):
                prompt = self.processor.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            else:
                prompt = query
            
            # Process inputs
            inputs = self.processor(
                text=prompt,
                images=processed_images if processed_images else None,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048
            )
            
            # Move to device
            for key in inputs:
                if isinstance(inputs[key], torch.Tensor):
                    inputs[key] = inputs[key].to(self.device, non_blocking=True)
            
            # Generate
            with torch.inference_mode():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=False,
                    temperature=0.4,
                    top_p=0.9,
                    repetition_penalty=1.05, # small guard against loops
                    pad_token_id=self.processor.tokenizer.pad_token_id,
                    eos_token_id=self.processor.tokenizer.eos_token_id,
                    use_cache=False
                )
            
            # Decode response
            if hasattr(self.processor, 'apply_chat_template'):
                full_response = self.processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
                
                # Extract model response
                if "<start_of_turn>model" in full_response:
                    response = full_response.split("<start_of_turn>model")[-1].strip()
                else:
                    input_length = inputs['input_ids'].shape[1]
                    generated_tokens = outputs[0][input_length:]
                    response = self.processor.tokenizer.decode(
                        generated_tokens, 
                        skip_special_tokens=True
                    ).strip()
            else:
                input_length = inputs['input_ids'].shape[1]
                generated_tokens = outputs[0][input_length:]
                response = self.processor.tokenizer.decode(
                    generated_tokens,
                    skip_special_tokens=True
                ).strip()
            
            # Cleanup
            del inputs, outputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            return response or "Generated response is empty."
            
        except Exception as e:
            logger.error(f"MedGemma generation failed: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return f"Generation error: {str(e)}"

# Initialize MedGemma
medgemma = GPUOptimizedMedGemma()

# ============================================================================
# SECTION 4: Multimodal Query System
# ============================================================================

def is_leishmania_related(text: str) -> bool:
    """Check if text is related to Leishmania."""
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in LEISHMANIA_KEYWORDS)

# --- Thresholds ---
RETRIEVAL_IP_THRESHOLD = 0.18     # inner-product on L2-normalized AVG vectors
SUPPORT_MAXSIM_THRESHOLD = 12.0   # sum of per-token MaxSim (tune below)
RETRY_SUPPORT_FRACTION = 0.9
SUPPORT_MIN = 8.0  # don't go below this

def _compute_maxsim_support(answer_text: str, top_hits: list, encoder: ColQwen2QueryEncoder) -> float:
    """
    Crude grounding score: encode answer with ColQwen2 text tokens (Q),
    take packed ColQwen2 tokens (P) of each hit via OFF/E, sum MaxSim over answer tokens,
    keep the best doc. Higher = more grounded.
    """
    if not answer_text.strip():
        return 0.0
    try:
        Q = encoder.encode_query_tokens(answer_text)  # [Tq, d], normalized
        E, OFF, dim = encoder._load_packed()
        best = 0.0
        for h in top_hits[:5]:  # check top 5
            rid = h.get("row_id")
            if rid is None or rid >= len(OFF): 
                continue
            start, length = OFF[rid]
            if start + length > E.shape[0]:
                continue
            P = torch.from_numpy(E[start:start+length]).to(Q.device)  # [Tp, d]
            # MaxSim (late interaction style): sum over answer tokens
            score = torch.max(Q @ P.T, dim=1).values.sum().item()
            if score > best: best = score
        return float(best)
    except Exception:
        return 0.0

def _is_leish_query_with_ctx(query: str, hits: list) -> bool:
    doc_ids = [h.get("doc_id","") for h in hits]
    text_lower = query.lower()
    kw_hit = any(k in text_lower for k in LEISHMANIA_KEYWORDS)
    ctx_hit = any("leish" in (d or "").lower() for d in doc_ids)
    return kw_hit or ctx_hit

def smart_query_system(query: str, query_images: Optional[List[str]] = None, 
                      top_k: int = 5, prioritize_leishmania: bool = False,
                      use_late_interaction: bool = True) -> Dict[str, Any]:
    """Enhanced query system using precomputed ColQwen2 embeddings."""
    
    start_time = time.time()
    query_images = query_images or []
    is_leishmania_query = is_leishmania_related(query)
    abstain_stage = None
    has_user_images = bool(query_images)
    try:
        # Stage 1: Retrieval using precomputed embeddings
        logger.info(f"Retrieving top {top_k} documents for: '{query[:50]}...'")
        
        use_late = bool(use_late_interaction and (is_leishmania_query or prioritize_leishmania))
        hits = query_encoder.search_late_interaction(query, k=top_k) if use_late else query_encoder.search_avg(query, k=top_k)
        
        if not hits:
            logger.warning("No documents retrieved")
            return {
                "text": "INSUFFICIENT_EVIDENCE",
                "images": [],
                "metadata": {
                    "query": query,
                    "processing_time": time.time() - start_time,
                    "sources_count": 0
                }
            }
        
        # NEW: Domain + strength gate
        max_ip = max(float(h.get("preselect_ip", h.get("score", 0.0))) for h in hits)
        is_leish = _is_leish_query_with_ctx(query, hits)

        # --- gating: only hard-abstain when off-topic AND weak ---
        weak = (max_ip < RETRIEVAL_IP_THRESHOLD)
        off_topic = (not is_leish) and (not has_user_images)

        if off_topic and weak:
            abstain_stage = "domain_or_strength"
            return {
                "text": "INSUFFICIENT_EVIDENCE",
                "images": [],
                "metadata": {
                    "query": query,
                    "processing_time": time.time() - start_time,
                    "sources_count": len(hits),
                    "retrieval_max_preselect_ip": max_ip,
                    "abstained": True,
                    "abstain_stage": abstain_stage,
                }
            }

        # If it's on-topic (or user gave images) but weak, widen and keep going
        if weak:
            wider_k = max(top_k, 32)
            hits = query_encoder.search_late_interaction(query, k=wider_k) if use_late else query_encoder.search_avg(query, k=wider_k)
        
        # Stage 2: Load retrieved images
        retrieved_images = []
        valid_hits = []
        
        for hit in hits:
            img_path = hit.get("image_path")
            if img_path and Path(img_path).exists():
                try:
                    img = Image.open(img_path).convert("RGB")
                    retrieved_images.append(img)
                    valid_hits.append(hit)
                except Exception as e:
                    logger.warning(f"Failed to load image {img_path}: {e}")
        
        # Add query images if provided
        all_images = []
        if query_images:
            for img_path in query_images:
                if Path(img_path).exists():
                    try:
                        img = Image.open(img_path).convert("RGB")
                        all_images.append(img)
                    except Exception as e:
                        logger.warning(f"Failed to load query image {img_path}: {e}")
        
        all_images.extend(retrieved_images)
        
        # NEW: Stage 3: Grounded prompt instead of raw query
        grounded_prompt = make_grounded_prompt(query, valid_hits)
        
        # Stage 4 + post-check + retry, with cleanup
        try:
            answer = medgemma.generate_answer(grounded_prompt, all_images)
            support = _compute_maxsim_support(answer, valid_hits, query_encoder)

            if answer.strip() == "INSUFFICIENT_EVIDENCE" or support < SUPPORT_MAXSIM_THRESHOLD:
                if top_k < 10:
                    wider = query_encoder.search_late_interaction(query, k=10) if use_late else query_encoder.search_avg(query, k=10)

                    wider_images, wider_valid_hits = [], []
                    for hit in wider:
                        ip = hit.get("image_path")
                        if ip and Path(ip).exists():
                            try:
                                wider_images.append(Image.open(ip).convert("RGB"))
                                wider_valid_hits.append(hit)
                            except:
                                pass

                    all_images_wider = []
                    for ip in (query_images or []):
                        if Path(ip).exists():
                            try:
                                all_images_wider.append(Image.open(ip).convert("RGB"))
                            except:
                                pass
                    all_images_wider.extend(wider_images)
                    grounded_prompt_wider = make_grounded_prompt(query, wider_valid_hits)
                    answer = medgemma.generate_answer(grounded_prompt_wider, all_images_wider)
                    support = _compute_maxsim_support(answer, wider_valid_hits, query_encoder)
                    retry_thresh = max(SUPPORT_MIN, RETRY_SUPPORT_FRACTION * SUPPORT_MAXSIM_THRESHOLD)
                    if support < retry_thresh:
                        answer = "INSUFFICIENT_EVIDENCE"
                        abstain_stage = "support_check_after_retry" 
                        valid_hits = wider_valid_hits
        finally:
            for im in all_images:
                try: im.close()
                except: pass
            for im in locals().get("all_images_wider", []):
                try: im.close()
                except: pass
        
        # Format response
        processing_time = time.time() - start_time

        # Create metadata
        leishmania_sources = sum(1 for hit in valid_hits 
                            if is_leishmania_related(hit.get("doc_id", "")))

        source_info = f"\n\n📄 Sources: {len(valid_hits)} pages retrieved"
        if leishmania_sources > 0:
            source_info += f" ({leishmania_sources} Leishmania-specific)"
        source_info += f"\n⚡ Processing time: {processing_time:.2f}s"

        response_images = [
            {
                "path": hit.get("image_path", ""),
                "metadata": hit,
                "relevance_rank": hit.get("rank", i + 1)
            }
            for i, hit in enumerate(valid_hits)
        ]

        # Only append source_info if answer is not INSUFFICIENT_EVIDENCE
        final_answer = answer + source_info if answer.strip() != "INSUFFICIENT_EVIDENCE" else answer
        # mark if we abstained here
        abstained_now = (final_answer.strip() == "INSUFFICIENT_EVIDENCE")
        if abstained_now and abstain_stage is None:
            abstain_stage = "support_check_after_retry"

        metadata = {
            "query": query,
            "query_images": query_images,
            "processing_time": processing_time,
            "leishmania_related": is_leishmania_query,
            "sources_count": len(valid_hits),
            "leishmania_sources": leishmania_sources,
            "retrieval_method": "late_interaction" if use_late else "averaged",
            "retrieval_max_preselect_ip": max_ip,
            "retrieval_maxsim_best": max(float(h["score"]) for h in hits),
            "abstained": abstained_now,
            "abstain_stage": abstain_stage,                                 
            "query_is_leish": is_leishmania_query,                         
            "context_has_leish": is_leish,                                  
        }
        return {
            "text": final_answer,
            "images": response_images,
            "metadata": metadata,
            "query_is_leish": is_leishmania_query,        # text-only view
            "context_has_leish": is_leish,                # retrieved-docs view
            "abstained": abstained_now,
            "abstain_stage": abstain_stage,
        }
        
    except Exception as e:
        logger.error(f"Error in smart_query_system: {e}", exc_info=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        return {
            "text": f"An error occurred: {str(e)}",
            "images": [],
            "metadata": {
                "query": query,
                "error": str(e),
                "processing_time": time.time() - start_time
            }
        }

# ============================================================================
# SECTION 5: Response Display and Management
# ============================================================================

def display_multimodal_response(result: Dict[str, Any]):
    """Display a multimodal response with proper formatting."""
    print("\n" + "="*70)
    print("           MULTIMODAL RESPONSE")
    print("="*70)
    
    # Display text response
    print(f"💬 Text Response:")
    print(f"{result.get('text', 'No text response available.')}")
    
    # Display images if available
    if result.get('images'):
        print(f"\n🖼️ Visual Evidence ({len(result['images'])} images):")
        print("-" * 50)
        for i, img_info in enumerate(result['images'], 1):
            print(f"  📸 Image {i} (Rank {img_info.get('relevance_rank', 'N/A')}): {os.path.basename(img_info.get('path', ''))}")
            meta = img_info.get('metadata', {})
            print(f"     Source: {meta.get('doc_id', 'unknown')} | Page: {meta.get('page', 'unknown')}")
    
    # Display metadata
    metadata = result.get('metadata', {})
    if metadata:
        print(f"\n📊 Processing Metadata:")
        print(f"   - Query: '{metadata.get('query', 'N/A')}'")
        if metadata.get('query_images'): 
            print(f"   - Query Images: {len(metadata.get('query_images', []))}")
        print(f"   - Processing time: {metadata.get('processing_time', 0):.2f}s")
        print(f"   - Leishmania-related: {metadata.get('leishmania_related', False)}")
        print(f"   - Retrieval method: {metadata.get('retrieval_method', 'unknown')}")
        print(f"   - Sources: {metadata.get('sources_count', 0)} ({metadata.get('leishmania_sources', 0)} Leishmania-specific)")
    print("="*70)

def save_multimodal_response(result: Dict[str, Any], output_dir: Path = None) -> str:
    """Save a multimodal response to files."""
    if output_dir is None:
        output_dir = A_RAG_DIR / "output"
    
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = int(time.time())
    
    try:
        # Save text response and metadata
        response_file = output_dir / f"response_{timestamp}.json"
        with open(response_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        # Copy relevant images
        if result.get('images'):
            img_dir = output_dir / f"response_images_{timestamp}"
            img_dir.mkdir(exist_ok=True)
            
            import shutil
            for img_info in result['images']:
                src_path = Path(img_info['path'])
                if src_path.exists():
                    dst_path = img_dir / src_path.name
                    shutil.copy2(src_path, dst_path)
                    
            logger.info(f"✅ Response and {len(result['images'])} images saved to {output_dir}")
        else:
            logger.info(f"✅ Response saved to {response_file}")
            
        return str(response_file)
        
    except Exception as e:
        logger.error(f"Error saving response: {e}")
        return None

# ============================================================================
# SECTION 6: Testing and Validation System
# ============================================================================

def run_system_tests():
    """Run comprehensive system tests."""
    print("\n" + "="*60)
    print("     GPU-OPTIMIZED MULTIMODAL RAG SYSTEM - TEST SUITE")
    print("="*60 + "\n")
    
    # Check if index exists
    if not (COLQWEN2_DIR / "manifest.json").exists():
        print("⚠️ ColQwen2 index not found. Building index first...")
        build_colqwen2_index()
        print("✅ Index built successfully!")
    
    # Create test image if needed
    test_img_path = A_RAG_DIR / "test_lesion.png"
    if not test_img_path.exists():
        try:
            from PIL import Image, ImageDraw
            img = Image.new('RGB', (300, 200), color='pink')
            draw = ImageDraw.Draw(img)
            draw.ellipse((100, 50, 200, 150), fill='red', outline='darkred')
            draw.text((10, 10), "Test Skin Lesion", fill="black")
            img.save(test_img_path)
            print(f"🖼️ Created test image: {test_img_path}")
        except Exception as e:
            print(f"Could not create test image: {e}")
            test_img_path = None
    
    # Test cases
    test_cases = [
        {
            "description": "Leishmania Text-Only Query",
            "query": "What are the clinical features of cutaneous leishmaniasis?",
            "query_images": None,
            "expected_leishmania": True
        },
        {
            "description": "General Medical Query",
            "query": "What are the symptoms of malaria?",
            "query_images": None,
            "expected_leishmania": False
        },
        {
            "description": "Multimodal Leishmania Query",
            "query": "Analyze this skin lesion for leishmaniasis signs.",
            "query_images": [str(test_img_path)] if test_img_path and test_img_path.exists() else None,
            "expected_leishmania": True
        }
    ]
    
    results = []
    
    for i, case in enumerate(test_cases, 1):
        print(f"\n--- Test Case {i}: {case['description']} ---")
        
        if case.get("query_images") and not case["query_images"][0]:
            print("⚠️ SKIPPING: Test image not available")
            continue
        
        try:
            start_time = time.time()
            
            result = smart_query_system(
                query=case['query'],
                query_images=case['query_images'],
                top_k=3
            )
            
            test_time = time.time() - start_time
            
            # Validate results
            success = True
            issues = []
            
            if not result.get('text'):
                success = False
                issues.append("No text response generated")
            
            metadata = result.get('metadata', {})
            if metadata.get('leishmania_related') != case['expected_leishmania']:
                issues.append(f"Leishmania detection mismatch: expected {case['expected_leishmania']}, got {metadata.get('leishmania_related')}")
            
            if metadata.get('sources_count', 0) == 0:
                issues.append("No sources retrieved")
            
            print(f"🔍 Query: '{case['query']}'")
            if case.get('query_images'):
                print(f"📷 Images: {len(case['query_images'])}")
            
            print(f"⏱️ Processing time: {test_time:.2f}s")
            print(f"📊 Sources: {metadata.get('sources_count', 0)}")
            print(f"🦠 Leishmania detected: {metadata.get('leishmania_related', False)}")
            print(f"🔍 Retrieval method: {metadata.get('retrieval_method', 'unknown')}")
            
            if success:
                print("✅ TEST PASSED")
            else:
                print("❌ TEST FAILED:")
                for issue in issues:
                    print(f"   - {issue}")
            
            results.append({
                'case': case['description'],
                'success': success,
                'time': test_time,
                'issues': issues
            })
            
        except Exception as e:
            print(f"❌ TEST ERROR: {e}")
            results.append({
                'case': case['description'],
                'success': False,
                'time': 0,
                'issues': [str(e)]
            })
        
        print("-" * 50)
    
    # Summary
    passed = sum(1 for r in results if r['success'])
    total = len(results)
    avg_time = sum(r['time'] for r in results) / len(results) if results else 0
    
    print(f"\n📊 TEST SUMMARY:")
    print(f"   Passed: {passed}/{total}")
    print(f"   Average processing time: {avg_time:.2f}s")
    
    if passed == total:
        print("🎉 ALL TESTS PASSED!")
    else:
        print("⚠️ Some tests failed. Check the issues above.")
    
    return results

# ============================================================================
# SECTION 7: Interactive System
# ============================================================================

def interactive_leishmania_rag():
    """Interactive multimodal RAG system."""
    print("\n" + "="*70)
    print("     INTERACTIVE MULTIMODAL LEISHMANIA RAG SYSTEM")
    print("="*70)
    print("🦠 Specialized for Leishmania research")
    print("🚀 GPU-accelerated with ColQwen2 + MedGemma")
    print("🖼️ Full multimodal support (text + images)")
    print("💡 Type 'help' for commands, 'quit' to exit")
    print("="*70 + "\n")
    
    # Check system readiness
    if not (COLQWEN2_DIR / "manifest.json").exists():
        print("⚠️ System not initialized. Building index...")
        build_colqwen2_index()
        print("✅ System ready!")
    
    while True:
        try:
            query = input("🔍 Your question (or 'help'/'quit'): ").strip()
            
            if not query:
                continue
                
            if query.lower() in ['quit', 'exit', 'q']:
                print("👋 Thank you for using the Multimodal Leishmania RAG system!")
                break
            
            if query.lower() == 'help':
                print("\n📚 Available commands:")
                print("  - Ask any medical question")
                print("  - After typing a question, you can add image paths")
                print("  - 'stats': Show system statistics")
                print("  - 'test': Run the test suite")
                print("  - 'rebuild': Rebuild the ColQwen2 index")
                print("  - 'quit': Exit the system")
                continue
            
            if query.lower() == 'stats':
                try:
                    with open(COLQWEN2_DIR / "manifest.json") as f:
                        manifest = json.load(f)
                    
                    print(f"\n📊 System Statistics:")
                    print(f"  - Indexed images: {manifest.get('n_images', 0):,}")
                    print(f"  - Token embeddings: {manifest.get('n_tokens_total', 0):,}")
                    print(f"  - Model: {manifest.get('model', 'unknown')}")
                    print(f"  - Embedding dimension: {manifest.get('dim', 'unknown')}")
                    print(f"  - Created: {manifest.get('created_at', 'unknown')}")
                except Exception as e:
                    print(f"❌ Could not load statistics: {e}")
                continue
            
            if query.lower() == 'test':
                run_system_tests()
                continue
            
            if query.lower() == 'rebuild':
                confirm = input("⚠️ Rebuild index? This will take time (y/n): ").strip().lower()
                if confirm == 'y':
                    build_colqwen2_index()
                    print("✅ Index rebuilt successfully!")
                continue
            
            # Handle multimodal input
            image_input = input("🖼️ Add image paths (optional, comma-separated): ").strip()
            query_images = []
            
            if image_input:
                for path in image_input.split(','):
                    p = Path(path.strip())
                    if p.exists() and p.is_file():
                        query_images.append(str(p))
                        print(f"  ✅ Added image: {p.name}")
                    else:
                        print(f"  ❌ Image not found: {p}")
            
            # Process query
            print(f"\n⏳ Processing query...")
            
            is_leish_query = is_leishmania_related(query)
            if is_leish_query:
                print("🦠 Leishmania-related query detected - using precise retrieval")
            
            result = smart_query_system(
                query=query,
                query_images=query_images,
                prioritize_leishmania=is_leish_query,
                use_late_interaction=is_leish_query
            )
            
            display_multimodal_response(result)
            
            # Offer to save
            save_option = input("\n💾 Save this response? (y/n): ").strip().lower()
            if save_option == 'y':
                saved_path = save_multimodal_response(result)
                if saved_path:
                    print(f"✅ Response saved to: {saved_path}")
                else:
                    print("❌ Failed to save response")
            
        except KeyboardInterrupt:
            print("\n👋 Exiting...")
            break
        except Exception as e:
            print(f"❌ An error occurred: {e}")
            logger.debug("Full error:", exc_info=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue

# ============================================================================
# SECTION 8: Utility Functions and System Management
# ============================================================================

def cleanup_gpu_memory():
    """Clean up GPU memory and optimize for next operations."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
        
        memory_allocated = torch.cuda.memory_allocated(0) / (1024**3)
        memory_cached = torch.cuda.memory_reserved(0) / (1024**3)
        
        logger.info(f"GPU memory cleaned - Allocated: {memory_allocated:.2f}GB, Cached: {memory_cached:.2f}GB")
        return {"allocated_gb": memory_allocated, "cached_gb": memory_cached}
    return None

def show_system_summary():
    """Display comprehensive system summary."""
    print("\n" + "="*60)
    print("           SYSTEM SUMMARY")
    print("="*60)
    
    # Check index status
    if (COLQWEN2_DIR / "manifest.json").exists():
        try:
            with open(COLQWEN2_DIR / "manifest.json") as f:
                manifest = json.load(f)
            print(f"📊 Index: {manifest.get('n_images', 0):,} images, {manifest.get('n_tokens_total', 0):,} tokens")
        except Exception:
            print("📊 Index: Present but could not read manifest")
    else:
        print("📊 Index: Not built - run build_colqwen2_index() first")
    
    # GPU info
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        mem_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        mem_allocated = torch.cuda.memory_allocated(0) / (1024**3)
        print(f"🚀 GPU: {gpu_name} ({mem_total:.1f}GB total, {mem_allocated:.2f}GB allocated)")
    else:
        print("❌ GPU: Not available (using CPU)")
    
    print(f"🤖 Models: ColQwen2 (retrieval), MedGemma (generation)")
    print(f"🖼️ Multimodal: ✅ Input (text+image), ✅ Output (text+image)")
    print(f"🦠 Specialization: Leishmania research with intelligent filtering")
    print("="*60)

def batch_process_queries(queries: List[Dict[str, Any]], output_dir: Path = None) -> List[Dict[str, Any]]:
    """Process multiple queries in batch."""
    if output_dir is None:
        output_dir = A_RAG_DIR / "batch_output"
    
    output_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"Starting batch processing of {len(queries)} queries...")
    
    results = []
    
    for i, query_data in enumerate(queries, 1):
        text_query = query_data.get("query")
        image_paths = query_data.get("query_images", [])
        
        if not text_query:
            logger.warning(f"Skipping query {i} - no text provided")
            continue
        
        logger.info(f"Processing batch query {i}/{len(queries)}: '{text_query[:50]}...'")
        
        try:
            result = smart_query_system(
                query=text_query,
                query_images=image_paths,
                prioritize_leishmania=is_leishmania_related(text_query)
            )
            
            # Save individual result
            save_multimodal_response(result, output_dir)
            results.append(result)
            
        except Exception as e:
            logger.error(f"Error processing query {i}: {e}")
            results.append({
                "text": f"Error: {str(e)}",
                "images": [],
                "metadata": {"query": text_query, "error": str(e)}
            })
    
    logger.info(f"✅ Batch processing complete - {len(results)} results")
    return results

# ============================================================================
# SECTION 9: Main Entry Points and Initialization
# ============================================================================

def initialize_system():
    """Initialize the complete RAG system."""
    print("🚀 Initializing GPU-Optimized Multimodal RAG System...")
    
    # Check for existing index
    if not (COLQWEN2_DIR / "manifest.json").exists():
        print("📚 No existing index found. Building ColQwen2 index...")
        build_colqwen2_index()
        print("✅ Index built successfully!")
    else:
        print("✅ Using existing ColQwen2 index")
    
    # Load models lazily (they'll be loaded on first use)
    print("🤖 Models configured for lazy loading")
    
    show_system_summary()
    
    print("\n🎉 System initialization complete!")
    print("💡 Available functions:")
    print("  - interactive_leishmania_rag() - Start interactive mode")
    print("  - run_system_tests() - Run comprehensive tests")
    print("  - smart_query_system(query, images) - Direct query processing")
    print("  - build_colqwen2_index() - Rebuild the search index")

# Auto-run initialization if this is the main execution
if __name__ == "__main__":
    initialize_system()
else:
    # When imported as module, show quick status
    show_system_summary()
    print("\n💡 Run initialize_system() to set up the complete RAG pipeline")
    print("💡 Run interactive_leishmania_rag() to start the interactive interface")

2025-08-12 19:30:42,857 - INFO - Using device: cuda
2025-08-12 19:30:42,858 - INFO - Project directory: /media/pc1/Ubuntu/Extend_Data/ngoc
2025-08-12 19:30:42,858 - INFO - RAG directory: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base


🚀 Initializing GPU-Optimized Multimodal RAG System...
✅ Using existing ColQwen2 index
🤖 Models configured for lazy loading

           SYSTEM SUMMARY
📊 Index: 1,501 images, 681,127 tokens
🚀 GPU: NVIDIA GeForce RTX 3090 (23.7GB total, 0.00GB allocated)
🤖 Models: ColQwen2 (retrieval), MedGemma (generation)
🖼️ Multimodal: ✅ Input (text+image), ✅ Output (text+image)
🦠 Specialization: Leishmania research with intelligent filtering

🎉 System initialization complete!
💡 Available functions:
  - interactive_leishmania_rag() - Start interactive mode
  - run_system_tests() - Run comprehensive tests
  - smart_query_system(query, images) - Direct query processing
  - build_colqwen2_index() - Rebuild the search index


generating answer

In [2]:
import json, os, time
from pathlib import Path
from typing import Dict, List, Optional, Any
from datetime import datetime
from tqdm import tqdm

class RAGAnswerGenerator:
    """
    Generates answers for a bank of questions using the existing multimodal RAG system.
    - Prefers qa_eval_1500.json (list of dicts with `question`, `question_id`, `source_file`, `chunk_id`, ...)
    - Can also read JSONL (falls back to fields `instruction`/`input` if present).
    - Reuses previously built ColQwen2 artifacts (meta.jsonl, FAISS, etc.).
    """

    def __init__(
        self,
        eval_file: Optional[str] = None,
        base_rag_dir: Optional[str] = None,
        default_num_questions: int = 50
    ):
        # Resolve base dirs from your existing code if available; else use your runtime paths
        self.A_RAG_DIR = Path(globals().get(
            "A_RAG_DIR",
            base_rag_dir or "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base"
        ))
        self.COLQWEN2_DIR = Path(globals().get(
            "COLQWEN2_DIR",
            self.A_RAG_DIR / "index" / "colqwen2"
        ))
        self.IMAGE_DIR = self.A_RAG_DIR / "images"

        # Evaluation file: prefer qa_eval_1500.json
        self.EVAL_FILE = Path(eval_file or (self.A_RAG_DIR / "qa_eval_1500.json"))

        # Optional chunk map (if you built it earlier). It’s OK if missing.
        self.CHUNK_DIR = self.A_RAG_DIR / "chunks"

        self.default_num_questions = default_num_questions

        self.evaluation_questions: List[Dict[str, Any]] = []
        self.chunk_data_map: Dict[str, Dict[str, Any]] = {}
        self._meta_by_doc: Dict[str, List[Dict[str, Any]]] = {}

        self._load_questions()
        self._load_chunk_data()
        self._load_meta_index()

        print(f"✅ RAGAnswerGenerator ready")
        print(f"   • Questions: {len(self.evaluation_questions)}")
        print(f"   • Chunks indexed: {len(self.chunk_data_map)}")
        print(f"   • Meta docs: {len(self._meta_by_doc)}")
        print(f"   • Image dir: {self.IMAGE_DIR}")

    # -----------------------------
    # Data loading
    # -----------------------------
    def _load_questions(self):
        if not self.EVAL_FILE.exists():
            # gentle fallback to your uploaded path, if you kept it there
            alt = Path("/mnt/data/qa_eval_1500.json")
            if alt.exists():
                self.EVAL_FILE = alt
            else:
                print(f"❌ Evaluation file not found: {self.EVAL_FILE}")
                return

        try:
            if self.EVAL_FILE.suffix.lower() == ".json":
                data = json.loads(self.EVAL_FILE.read_text(encoding="utf-8"))
                if isinstance(data, list):
                    self.evaluation_questions = data
                else:
                    print("⚠️ JSON root is not a list; wrapping as single-item list.")
                    self.evaluation_questions = [data]
            else:
                # JSONL fallback (e.g., qa_eval_1500_instruction.jsonl)
                items = []
                with open(self.EVAL_FILE, "r", encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        obj = json.loads(line)
                        # Map SFT-ish fields into our expected schema
                        q = obj.get("question") or obj.get("instruction") or obj.get("prompt")
                        if q and obj.get("input"):
                            q = f"{q}\n\n{obj['input']}"
                        if not q:
                            continue
                        items.append({
                            "question_id": obj.get("question_id") or obj.get("id") or f"q_{len(items)}",
                            "question": q,
                            "source_file": obj.get("source_file"),
                            "chunk_id": obj.get("chunk_id"),
                            "gold_answer": obj.get("answer") or obj.get("output")
                        })
                self.evaluation_questions = items

            print(f"📋 Loaded {len(self.evaluation_questions)} questions from {self.EVAL_FILE}")

        except Exception as e:
            print(f"❌ Error loading {self.EVAL_FILE}: {e}")
            self.evaluation_questions = []

    def _load_chunk_data(self):
        if not self.CHUNK_DIR.exists():
            return
        try:
            count = 0
            for p in self.CHUNK_DIR.glob("*.json"):
                try:
                    obj = json.loads(p.read_text(encoding="utf-8"))
                    if isinstance(obj, list):
                        for ch in obj:
                            cid = ch.get("chunk_id")
                            if cid:
                                self.chunk_data_map[cid] = ch
                                count += 1
                    elif isinstance(obj, dict) and obj.get("chunk_id"):
                        self.chunk_data_map[obj["chunk_id"]] = obj
                        count += 1
                except Exception:
                    continue
            if count:
                print(f"📚 Loaded {count} chunks from {self.CHUNK_DIR}")
        except Exception as e:
            print(f"⚠️ Could not load chunks: {e}")

    def _load_meta_index(self):
        """Build doc_id -> list[ {image_path, page, ...} ] from index/colqwen2/meta.jsonl (output from your encoder)."""
        meta_path = self.COLQWEN2_DIR / "meta.jsonl"
        if not meta_path.exists():
            print(f"⚠️ Meta file not found (no page images mapping): {meta_path}")
            return
        try:
            by_doc = {}
            with open(meta_path, "r", encoding="utf-8") as f:
                for line in f:
                    d = json.loads(line)
                    doc_id = d.get("doc_id")
                    if not doc_id:
                        # derive from image_path directory if needed
                        ip = d.get("image_path")
                        if ip:
                            doc_id = Path(ip).parent.name
                    if not doc_id:
                        continue
                    by_doc.setdefault(doc_id, []).append(d)
            # sort each doc's list by page if available
            for doc_id, rows in by_doc.items():
                by_doc[doc_id] = sorted(
                    rows,
                    key=lambda r: (r.get("page") if isinstance(r.get("page"), int) else 10**9,
                                   r.get("image_path",""))
                )
            self._meta_by_doc = by_doc
        except Exception as e:
            print(f"⚠️ Error reading meta.jsonl: {e}")

    # -----------------------------
    # Image selection
    # -----------------------------
    @staticmethod
    def _derive_doc_id(q: Dict[str, Any]) -> Optional[str]:
        sf = q.get("source_file")
        if sf:
            return Path(sf).stem
        cid = q.get("chunk_id")
        if cid and "_s" in cid:
            # e.g., "SomeDoc_s11_c0" -> "SomeDoc"
            return cid.split("_s", 1)[0]
        return None

    def _pick_doc_pages(self, doc_id: str, max_pages: int = 2) -> List[str]:
        """Pick 1–2 representative page images for the doc_id using your meta.jsonl mapping."""
        if not doc_id or doc_id not in self._meta_by_doc:
            return []
        rows = self._meta_by_doc[doc_id][:max_pages]
        paths = []
        for r in rows:
            p = r.get("image_path")
            if p and Path(p).exists():
                paths.append(p)
        return paths

    def find_contextual_image_for_question(self, q: Dict[str, Any]) -> List[str]:
        """
        Priority:
        1) If chunk map has `associated_images`, try those IDs (images/<id>.png|jpg|jpeg).
        2) Else derive doc_id from question and pick first 1–2 page images via meta.jsonl.
        """
        # 1) chunk-associated images (if you built them)
        cid = q.get("source_chunk_id") or q.get("chunk_id")
        if cid and cid in self.chunk_data_map:
            imgs = []
            for image_id in self.chunk_data_map[cid].get("associated_images", []):
                for ext in (".png", ".jpg", ".jpeg", ".webp"):
                    p = self.IMAGE_DIR / f"{image_id}{ext}"
                    if p.exists():
                        imgs.append(str(p))
                        break
            if imgs:
                return imgs

        # 2) doc-level page images
        doc_id = self._derive_doc_id(q)
        if doc_id:
            return self._pick_doc_pages(doc_id, max_pages=2)

        return []

    # -----------------------------
    # RAG Invocation
    # -----------------------------
    def answer_query(self, query: str, image_paths: Optional[List[str]] = None) -> Dict[str, Any]:
        if "smart_query_system" not in globals():
            return {
                "text": "Error: smart_query_system not available. Please import/run the RAG system first.",
                "images": [],
                "metadata": {"error": "smart_query_system missing"}
            }
        try:
            return smart_query_system(query=query, query_images=image_paths or [])
        except Exception as e:
            return {"text": f"Error: {e}", "images": [], "metadata": {"error": str(e)}}

    def display_response(self, response: Dict[str, Any]):
        if "display_multimodal_response" in globals():
            try:
                display_multimodal_response(response)
                return
            except Exception as e:
                print(f"⚠️ display_multimodal_response failed: {e}")
        # Fallback
        print("💬 Text:", response.get("text", ""))
        if response.get("images"):
            print(f"🖼️ {len(response['images'])} images attached")

    # -----------------------------
    # Generation loop
    # -----------------------------
    def run_generation(self, num_questions_to_run=None, save_dir=None,
                   stream_path=None, resume=True):
        out_dir = Path(save_dir or (self.A_RAG_DIR / "eval_outputs"))
        out_dir.mkdir(parents=True, exist_ok=True)

        if stream_path is None:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            stream_path = out_dir / f"rag_generated_answers_{ts}.jsonl"
        else:
            stream_path = Path(stream_path)

        # collect already-done ids if resuming
        done_ids = set()
        if resume and stream_path.exists():
            with open(stream_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        done_ids.add(json.loads(line)["question_id"])
                    except Exception:
                        pass

        qs = self.evaluation_questions[: (num_questions_to_run or self.default_num_questions)]
        results = []
        with open(stream_path, "a", encoding="utf-8") as f_jsonl:
            for i, q in enumerate(qs, 1):
                qid = q.get("question_id", f"q_{i}")
                if qid in done_ids:
                    continue  # already saved

                q_text = q.get("question") or q.get("instruction") or ""
                if q_text and q.get("input"):
                    q_text = f"{q_text}\n\n{q['input']}"

                ctx_imgs = self.find_contextual_image_for_question(q)
                t0 = time.time()
                resp = self.answer_query(q_text, image_paths=ctx_imgs)
                dt = time.time() - t0

                record = {
                    "question_id": qid,
                    "question_text": q_text,
                    "source_file": q.get("source_file"),
                    "chunk_id": q.get("chunk_id") or q.get("source_chunk_id"),
                    "gold_answer": q.get("answer") or q.get("gold_answer") or q.get("output"),
                    "context_images_used": ctx_imgs,
                    "rag_answer": resp.get("text", ""),
                    "retrieved_contexts": [
                        {"path": img.get("path",""), "metadata": img.get("metadata",{}),
                        "relevance_rank": img.get("relevance_rank", 0)}
                        for img in (resp.get("images") or [])
                    ],
                    "generation_metadata": {
                        "generation_time_seconds": dt,
                        "timestamp": datetime.now().isoformat(),
                        "rag_system_metadata": resp.get("metadata", {})
                    }
                }

                # stream immediately
                f_jsonl.write(json.dumps(record, ensure_ascii=False) + "\n")
                f_jsonl.flush()  # optional: ensure it hits disk now

                results.append(record)

        # keep your summary JSON too
        self._save_results(results, save_dir=out_dir)


    def _save_results(self, results: List[Dict[str, Any]], save_dir: Optional[str] = None):
        out_dir = Path(save_dir or (self.A_RAG_DIR / "eval_outputs"))
        out_dir.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = out_dir / f"rag_generated_answers_{ts}.json"

        payload = {
            "metadata": {
                "generation_timestamp": datetime.now().isoformat(),
                "total_questions_processed": len(results),
                "successful_generations": sum(1 for r in results if "error" not in r.get("generation_metadata", {})),
                "failed_generations": sum(1 for r in results if "error" in r.get("generation_metadata", {})),
                "eval_file": str(self.EVAL_FILE),
                "rag_workspace": str(self.A_RAG_DIR),
                "colqwen2_dir": str(self.COLQWEN2_DIR),
            },
            "generated_answers": results
        }

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

        size_mb = out_path.stat().st_size / (1024 * 1024)
        print(f"\n💾 Saved results: {out_path} ({size_mb:.1f} MB)")

In [3]:
from pathlib import Path
import numpy as np, faiss

A_RAG_DIR = Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base")
COLQWEN2_DIR = A_RAG_DIR / "index" / "colqwen2"
INDEX_DIR = A_RAG_DIR / "index"

avg_path   = COLQWEN2_DIR / "avg_vectors.npy"
faiss_path = INDEX_DIR / "image_avg.faiss"

assert avg_path.exists(), f"Missing {avg_path}"
AVG = np.load(avg_path).astype("float32")
AVG /= (np.linalg.norm(AVG, axis=1, keepdims=True) + 1e-12)

index = faiss.IndexFlatIP(AVG.shape[1])
index.add(AVG)
faiss.write_index(index, str(faiss_path))

print(f"✅ Built FAISS: {faiss_path} with {AVG.shape[0]} vectors (dim={AVG.shape[1]})")

✅ Built FAISS: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/index/image_avg.faiss with 1501 vectors (dim=128)


In [ ]:
# driver_restart_clean.py (minimal)
from pathlib import Path
import json, time
EVAL_FILE = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/evaluation/qa_sets/qa_eval_1500.json"
OUT_DIR   = A_RAG_DIR / "eval_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
STREAM    = OUT_DIR / f"rag_generated_answers_restart_{int(time.time())}.jsonl"

gen = RAGAnswerGenerator(eval_file=EVAL_FILE)

# Fresh run from 1..N, but resumable if kernel dies mid-way
gen.run_generation(
    num_questions_to_run=1500,
    save_dir=str(OUT_DIR),
    stream_path=str(STREAM),
    resume=True   # resume against the same STREAM if interrupted
)

📋 Loaded 1500 questions from /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/evaluation/qa_sets/qa_eval_1500.json


2025-08-12 19:31:05,567 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-12 19:31:05,570 - INFO - Loaded FAISS index: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/index/image_avg.faiss
2025-08-12 19:31:05,575 - INFO - Loaded metadata for 1501 images


📚 Loaded 5892 chunks from /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/chunks
✅ RAGAnswerGenerator ready
   • Questions: 1500
   • Chunks indexed: 5892
   • Meta docs: 211
   • Image dir: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images


/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-08-12 19:31:13,375 - INFO - Loading ColQwen2 query encoder: vidore/colqwen2-v1.0-hf
2025-08-12 19:31:14,449 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
2025-08-12 19:31:46,587 - INFO - Loading MedGemma: google/medgemma-4b-it
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
2025-08-12 19:31:55,838 - INFO - We will use 90% of th

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2025-08-12 19:32:44,769 - INFO - ✅ MedGemma loaded on cuda
2025-08-12 19:32:44,770 - INFO - Generating answer with 3 images
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2025-08-12 19:44:27,700 - INFO - Loaded packed embeddings: (681127, 128)
2025-08-12 19:44:27,790 - INFO - Retrieving top 5 documents for: 'How is this condition defined?...'
2025-08-12 19:44:27,884 - INFO - Generating answer with 3 images
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2025-08-12 19:55:48,575 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-12 19:55:48,711 - INFO - Generating answer with 3 images
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2025-08-12 20:06:56,884 - INFO - Ret

In [7]:
# Point to your chosen eval set (recommended)
EVAL_FILE = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/evaluation/qa_sets/qa_eval_1500.json"
gen = RAGAnswerGenerator(eval_file=EVAL_FILE)
gen.run_generation(num_questions_to_run=1500)  # or any number 

📋 Loaded 1500 questions from /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/evaluation/qa_sets/qa_eval_1500.json


2025-08-12 19:08:23,332 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-12 19:08:23,375 - INFO - Retrieving top 5 documents for: 'How is this condition defined?...'
2025-08-12 19:08:23,404 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-12 19:08:23,434 - INFO - Retrieving top 5 documents for: 'What are the key characteristics of this disease?...'
2025-08-12 19:08:23,463 - INFO - Retrieving top 5 documents for: 'What are the key characteristics of this disease?...'
2025-08-12 19:08:23,496 - INFO - Retrieving top 5 documents for: 'What are the key characteristics of this disease?...'


📚 Loaded 5892 chunks from /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/chunks
✅ RAGAnswerGenerator ready
   • Questions: 1500
   • Chunks indexed: 5892
   • Meta docs: 211
   • Image dir: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images


2025-08-12 19:08:23,527 - INFO - Retrieving top 5 documents for: 'How is this condition defined?...'
2025-08-12 19:08:23,567 - INFO - Retrieving top 5 documents for: 'How is this condition defined?...'
2025-08-12 19:08:23,596 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-12 19:08:23,626 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-12 19:08:23,655 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-12 19:08:23,684 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-12 19:08:23,714 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-12 19:08:23,743 - INFO - Retrieving top 5 documents for: 'What are the key characteristics of this disease?...'
2025-08-12 19:08:23,773 - INFO - Retrieving top 5 documents for: 'What are the key characteristics of this disease?...'
2025-08-12 19:08:23,802

KeyboardInterrupt: 

In [4]:
# Point to your chosen eval set (recommended)
EVAL_FILE = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/evaluation/qa_sets/qa_eval_1500.json"
gen = RAGAnswerGenerator(eval_file=EVAL_FILE)
gen.run_generation(num_questions_to_run=1500)  # or any number 

📋 Loaded 1500 questions from /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/evaluation/qa_sets/qa_eval_1500.json
📚 Loaded 5892 chunks from /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/chunks
✅ RAGAnswerGenerator ready
   • Questions: 1500
   • Chunks indexed: 5892
   • Meta docs: 211
   • Image dir: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images

🚀 Generating answers for 1500 questions


Generating:   0%|          | 0/1500 [00:00<?, ?it/s]2025-08-11 22:25:25,361 - INFO - Retrieving top 5 documents for: 'What is the disease being described?...'
2025-08-11 22:25:25,363 - INFO - Loaded FAISS index: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/index/image_avg.faiss
2025-08-11 22:25:25,369 - INFO - Loaded metadata for 1501 images



📝 [1/1500] 1-case-Visceral leishmaniasis as a possible reason for pancytopenia_s8_c0_qa_0
❓ What is the disease being described?
🖼️ Context pages: p0002_xref4.jpeg, p0002_xref7.jpeg


/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-08-11 22:25:33,182 - INFO - Loading ColQwen2 query encoder: vidore/colqwen2-v1.0-hf
2025-08-11 22:25:34,078 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
2025-08-11 22:26:05,519 - INFO - Generating answer with 7 images
2025-08-11 22:26:05,567 - INFO - Loading MedGemma: google/medgemma-4b-it
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fa

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...


           MULTIMODAL RESPONSE
💬 Text Response:
Based on the provided images, the disease appears to be **squamous cell carcinoma**.

Here's a breakdown of the key characteristics based on the microscopic image:

*   **Cellular Morphology:** The cells are characterized by their irregular shape, enlarged nuclei, and prominent nucleoli. This indicates malignant transformation.
*   **Increased Mitotic Activity:** The presence of multiple cells undergoing mitosis (cell division) suggests rapid cell proliferation, a hallmark of cancer.
*   **Desmoplasia:** The presence of a dense, fibrous stroma (connective tissue) surrounding the tumor cells is called desmoplasia. This is a common feature of many carcinomas and contributes to tumor growth and spread.
*   **Keratinization:** The presence of keratin pearls (circular arrangements of keratinized cells) is a characteristic feature of squamous cell carcinoma.

**In summary, the key characteristics of the disease are:**

*   Irregularly shaped c

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Generating:   9%|▉         | 140/1500 [12:20:29<123:52:17, 327.89s/it]2025-08-12 10:45:54,596 - INFO - Retrieving top 5 documents for: 'How is this condition defined?...'
2025-08-12 10:45:54,637 - INFO - Generating answer with 7 images
2025-08-12 10:45:54,637 - INFO - Generating answer with 3 images
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



           MULTIMODAL RESPONSE
💬 Text Response:
Based on the images and the provided information, the condition appears to be **basal cell carcinoma (BCC)**.

Here's why:

*   **Appearance:** BCCs are typically pearly or waxy, with rolled borders. They can also be flat and flesh-colored or brown. The images show a lesion with a raised, irregular surface, which is consistent with BCC.
*   **Location:** BCCs are most commonly found on sun-exposed areas, such as the face, ears, neck, and scalp. The location in the images is consistent with this.
*   **Other possibilities:** While other skin cancers could be considered, the description and appearance are most suggestive of BCC.

It's important to note that a definitive diagnosis can only be made by a qualified medical professional through a clinical examination and potentially a biopsy.

Final Answer: The final answer is $\boxed{basal cell carcinoma}$

📄 Sources: 5 pages retrieved (5 Leishmania-specific)
⚡ Processing time: 330.22s

🖼️ Vis

Generating:   9%|▉         | 140/1500 [12:21:23<120:02:01, 317.74s/it]


KeyboardInterrupt: 